# Aquatic Inpainting v5: all-train DINO/VQ weighted KNN

Metric-aligned solution: run official `dino_vq.py` over every train image, run it over every test image, then predict hidden patch codes from all train examples using nearby visible-code matches weighted by grid distance. No RGB MAE; optimize the submitted code IDs directly.



In [ ]:
import json
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as exc:
    _HF_SECRET_ERROR = repr(exc)
else:
    _HF_SECRET_ERROR = ""
import random
import shutil
import subprocess
import sys
import time
import zipfile
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch

@dataclass
class Config:
    version: str = 'v5'
    seed: int = 5364
    max_train_images: int = int(os.environ.get('MAX_TRAIN_IMAGES', '0'))  # 0 = all train
    dino_batch_size: int = int(os.environ.get('DINO_BATCH_SIZE', '32'))
    sample_images: int = 10
    alpha: float = float(os.environ.get('KNN_ALPHA', '0.65'))

CFG = Config()
random.seed(CFG.seed)
np.random.seed(CFG.seed)

WORKING = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
OUT = WORKING / f'aquatic_prior_{CFG.version}'
OUT.mkdir(parents=True, exist_ok=True)
RUN_SUMMARY_PATH = OUT / 'run_summary.json'
progress = {'status':'running','stage':'start','config':asdict(CFG),'artifacts':[],'warnings':[]}
if _HF_SECRET_ERROR:
    progress['warnings'].append('Initial HF_TOKEN Kaggle Secret read failed: ' + _HF_SECRET_ERROR[:200])
def save_progress():
    RUN_SUMMARY_PATH.write_text(json.dumps(progress, indent=2, default=str), encoding='utf-8')
def load_hf_token():
    secret_names = [
        'HF_TOKEN', 'HUGGING_FACE_HUB_TOKEN', 'HF_HUB_TOKEN',
        'HUGGINGFACE_TOKEN', 'HUGGING_FACE_TOKEN', 'HUGGINGFACEHUB_API_TOKEN',
    ]
    for name in secret_names:
        token = os.environ.get(name)
        if token:
            progress['hf_token_source'] = f'env:{name}'
            return token
    errors = []
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for name in secret_names:
            try:
                token = client.get_secret(name)
                if token:
                    progress['hf_token_source'] = f'kaggle_secret:{name}'
                    return token
            except Exception as exc:
                errors.append(f'{name}: {repr(exc)[:120]}')
    except Exception as exc:
        errors.append(f'kaggle_secrets import/client: {repr(exc)[:120]}')
    progress['warnings'].append('HF token unavailable from env/Kaggle Secrets names tried: ' + '; '.join(errors[:4]))
    return None

def export_hf_token(token):
    if not token:
        return
    for name in ['HF_TOKEN', 'HUGGING_FACE_HUB_TOKEN', 'HF_HUB_TOKEN']:
        os.environ[name] = token

HF_TOKEN = load_hf_token()
export_hf_token(HF_TOKEN)
progress['hf_token_available'] = bool(HF_TOKEN)
save_progress()
print({'out': str(OUT), 'max_train_images': CFG.max_train_images, 'hf_token_available': bool(HF_TOKEN)})



## Resolve dataset paths


In [ ]:

REQUIRED = {'train', 'test', 'target.csv', 'dino_vq.py', 'codebook.npy'}
IMAGE_SUFFIXES = {'.png', '.jpg', '.jpeg', '.bmp', '.webp'}

def find_dataset_root(base=Path('/kaggle/input')):
    candidates = []
    if base.exists():
        for target in base.rglob('target.csv'):
            root = target.parent
            names = {p.name for p in root.iterdir()}
            score = len(REQUIRED & names)
            if score >= 3:
                candidates.append((score, root))
    if not candidates:
        raise FileNotFoundError('dataset root not found')
    candidates.sort(reverse=True, key=lambda item: item[0])
    return candidates[0][1]

DATA = find_dataset_root()
TRAIN_DIR = DATA / 'train'
TEST_DIR = DATA / 'test'
TARGET_CSV = DATA / 'target.csv'
DINO_VQ = DATA / 'dino_vq.py'
CODEBOOK = DATA / 'codebook.npy'
for p in [TRAIN_DIR, TEST_DIR, TARGET_CSV, DINO_VQ, CODEBOOK]:
    if not p.exists():
        raise FileNotFoundError(str(p))
progress.update({'stage':'paths_ready','dataset_root':str(DATA)})
save_progress()
print({'dataset_root': str(DATA)})


## Parse target.csv and train image list


In [ ]:
def parse_competition_target(path):
    raw = pd.read_csv(path)
    id_col = 'Id' if 'Id' in raw.columns else 'ID' if 'ID' in raw.columns else None
    patch_col = 'patch_index' if 'patch_index' in raw.columns else 'target' if 'target' in raw.columns else 'Target' if 'Target' in raw.columns else None
    if id_col is None:
        raise ValueError(f'No Id/ID column in {list(raw.columns)}')
    rows = []
    if raw[id_col].astype(str).str.contains('_').all():
        for source_id in raw[id_col].astype(str):
            image_id, patch = source_id.rsplit('_', 1)
            rows.append({'image_id': image_id, 'patch_index': int(patch), 'Id': source_id})
    elif patch_col is not None:
        for image_id, raw_indices in zip(raw[id_col].astype(str), raw[patch_col]):
            parts = raw_indices.replace(',', ' ').split() if isinstance(raw_indices, str) else [raw_indices]
            for part in parts:
                patch = int(part)
                rows.append({'image_id': image_id, 'patch_index': patch, 'Id': f'{image_id}_{patch}'})
    else:
        raise ValueError(f'Cannot parse target columns {list(raw.columns)}')
    df = pd.DataFrame(rows)
    if df.empty or not df['patch_index'].between(0, 63).all():
        raise ValueError('bad target patch rows')
    return df

def list_images(folder):
    return sorted([p for p in folder.rglob('*') if p.suffix.lower() in IMAGE_SUFFIXES])

target_df = parse_competition_target(TARGET_CSV)
all_train_images = list_images(TRAIN_DIR)
train_images = all_train_images if CFG.max_train_images <= 0 else all_train_images[:CFG.max_train_images]
test_images = list_images(TEST_DIR)
if not train_images or not test_images:
    raise RuntimeError('missing train/test images')
progress.update({'stage':'target_ready','target_rows':len(target_df),'train_images_used':len(train_images),'test_images':len(test_images)})
save_progress()
print({'target_rows': len(target_df), 'train_images_used': len(train_images), 'test_images': len(test_images)})



## Build train target CSV for DINO/VQ


In [ ]:
train_target_rows = []
all_patches = ' '.join(str(i) for i in range(64))
for p in train_images:
    train_target_rows.append({'ID': p.stem, 'target': all_patches})
train_target_csv = WORKING / 'train_target_v5.csv'
pd.DataFrame(train_target_rows).to_csv(train_target_csv, index=False)
progress['train_target_csv'] = str(train_target_csv)
save_progress()
print({'train_target_rows': len(train_target_rows), 'patch_labels_requested': len(train_target_rows) * 64})



## Run official DINO/VQ on all train images



In [ ]:
progress['stage'] = 'dino_train_codes'
save_progress()
train_codes_csv = OUT / 'train_codes_v5.csv'
if not HF_TOKEN:
    progress['dino_train_returncode'] = None
    progress['warnings'].append('Skipping train DINO/VQ because HF_TOKEN is unavailable in this Kaggle run.')
    save_progress()
    print({'train_codes_csv': None, 'skipped': 'no_hf_token'})
else:
    cmd = [
        sys.executable, str(DINO_VQ),
        '--images-root', str(TRAIN_DIR),
        '--target-csv', str(train_target_csv),
        '--codebook-path', str(CODEBOOK),
        '--output-csv', str(train_codes_csv),
        '--batch-size', str(CFG.dino_batch_size),
    ]
    print({'cmd': ' '.join(cmd)})
    result = subprocess.run(cmd, cwd=str(DATA), text=True, capture_output=True, timeout=14400, env=os.environ.copy())
    progress['dino_train_returncode'] = result.returncode
    progress['dino_train_stdout_tail'] = result.stdout[-2000:]
    progress['dino_train_stderr_tail'] = result.stderr[-4000:]
    if result.returncode != 0:
        progress['warnings'].append('Train DINO/VQ failed; falling back. stderr tail: ' + result.stderr[-1000:])
    elif train_codes_csv.exists():
        progress['artifacts'].append(str(train_codes_csv))
    else:
        progress['warnings'].append(f'Train DINO/VQ returned 0 but missing {train_codes_csv}; falling back.')
    save_progress()
    print({'train_codes_csv': str(train_codes_csv) if train_codes_csv.exists() else None, 'returncode': result.returncode})



## Run official DINO/VQ on test images



In [ ]:
progress['stage'] = 'dino_test_codes'
save_progress()
all_patches = ' '.join(str(i) for i in range(64))
test_target_rows = [{'ID': p.stem, 'target': all_patches} for p in test_images]
test_target_csv = WORKING / 'test_all_target_v5.csv'
pd.DataFrame(test_target_rows).to_csv(test_target_csv, index=False)
test_codes_csv = OUT / 'test_codes_v5.csv'
if not HF_TOKEN:
    progress['dino_test_returncode'] = None
    progress['warnings'].append('Skipping test DINO/VQ because HF_TOKEN is unavailable in this Kaggle run.')
    save_progress()
    print({'test_codes_csv': None, 'skipped': 'no_hf_token'})
else:
    cmd = [
        sys.executable, str(DINO_VQ),
        '--images-root', str(TEST_DIR),
        '--target-csv', str(test_target_csv),
        '--codebook-path', str(CODEBOOK),
        '--output-csv', str(test_codes_csv),
        '--batch-size', str(CFG.dino_batch_size),
    ]
    print({'cmd': ' '.join(cmd)})
    result = subprocess.run(cmd, cwd=str(DATA), text=True, capture_output=True, timeout=7200, env=os.environ.copy())
    progress['dino_test_returncode'] = result.returncode
    progress['dino_test_stdout_tail'] = result.stdout[-2000:]
    progress['dino_test_stderr_tail'] = result.stderr[-4000:]
    if result.returncode != 0:
        progress['warnings'].append('Test DINO/VQ failed; falling back. stderr tail: ' + result.stderr[-1000:])
    elif test_codes_csv.exists():
        progress['artifacts'].append(str(test_codes_csv))
    else:
        progress['warnings'].append(f'Test DINO/VQ returned 0 but missing {test_codes_csv}; falling back.')
    save_progress()
    print({'test_codes_csv': str(test_codes_csv) if test_codes_csv.exists() else None, 'returncode': result.returncode})



## All-train weighted KNN predictor on GPU



In [ ]:
def normalize_submission_cols(df):
    rename = {}
    if 'ID' in df.columns and 'Id' not in df.columns:
        rename['ID'] = 'Id'
    if 'target' in df.columns and 'Target' not in df.columns:
        rename['target'] = 'Target'
    return df.rename(columns=rename)

def split_code_id(value):
    image_id, patch = str(value).rsplit('_', 1)
    return image_id, int(patch)

def codes_to_matrix(csv_path):
    df = normalize_submission_cols(pd.read_csv(csv_path))
    if not {'Id', 'Target'} <= set(df.columns):
        raise ValueError(f'bad code columns {list(df.columns)}')
    parsed = df['Id'].map(split_code_id)
    df['image_id'] = [x[0] for x in parsed]
    df['patch_index'] = [x[1] for x in parsed]
    if not df['Target'].between(0, 1023).all():
        raise ValueError(f'code outside 0..1023 in {csv_path}')
    image_ids = sorted(df['image_id'].unique())
    mat = np.full((len(image_ids), 64), -1, dtype=np.int16)
    pos = {image_id: i for i, image_id in enumerate(image_ids)}
    for row in df.itertuples(index=False):
        mat[pos[row.image_id], int(row.patch_index)] = int(row.Target)
    if (mat < 0).any():
        raise ValueError(f'incomplete code matrix {csv_path}')
    return image_ids, mat

train_ids, train_mat = codes_to_matrix(train_codes_csv)
test_ids, test_mat = codes_to_matrix(test_codes_csv)
test_pos = {image_id: i for i, image_id in enumerate(test_ids)}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
train_codes_t = torch.as_tensor(train_mat, dtype=torch.int16, device=device)
coords = torch.tensor([(i // 8, i % 8) for i in range(64)], dtype=torch.float32, device=device)
progress['predictor_source'] = f'all_train_weighted_knn_alpha_{CFG.alpha}'
progress['knn_device'] = str(device)

def weighted_knn_prediction(image_id, patch_indices):
    if image_id not in test_pos:
        raise KeyError(f'missing test codes for {image_id}')
    hidden = sorted(map(int, patch_indices))
    hidden_set = set(hidden)
    visible = [i for i in range(64) if i not in hidden_set]
    visible_t = torch.tensor(visible, dtype=torch.long, device=device)
    test_visible = torch.as_tensor(test_mat[test_pos[image_id], visible], dtype=torch.int16, device=device)
    matches = (train_codes_t[:, visible_t] == test_visible)
    preds = {}
    scores_seen = []
    for patch in hidden:
        dist = torch.abs(coords[visible_t] - coords[patch]).sum(dim=1)
        weights = torch.pow(torch.tensor(CFG.alpha, dtype=torch.float32, device=device), dist)
        scores = matches.float().matmul(weights)
        best_idx = int(torch.argmax(scores).item())
        preds[patch] = int(train_mat[best_idx, patch])
        scores_seen.append(float(scores[best_idx].item()))
    return preds, float(np.mean(scores_seen)) if scores_seen else 0.0

nn_debug = []
for image_id, group in target_df.groupby('image_id'):
    preds, score = weighted_knn_prediction(image_id, group['patch_index'].tolist())
    nn_debug.append({'image_id': image_id, 'weighted_visible_score': score})

nn_debug_df = pd.DataFrame(nn_debug)
prior_path = OUT / 'weighted_knn_scores_v5.csv'
nn_debug_df.to_csv(prior_path, index=False)
progress['artifacts'].append(str(prior_path))
progress['mean_weighted_visible_score'] = float(nn_debug_df['weighted_visible_score'].mean())
save_progress()
print(nn_debug_df.head(10).to_string(index=False))
print({'mean_weighted_visible_score': progress['mean_weighted_visible_score'], 'train_codes': train_mat.shape, 'test_codes': test_mat.shape, 'device': str(device)})



## Write v5 submission and 10 visual samples



In [ ]:
rows = []
for image_id, group in target_df.groupby('image_id'):
    patch_indices = group['patch_index'].astype(int).tolist()
    preds, _ = weighted_knn_prediction(image_id, patch_indices)
    for row in group.itertuples(index=False):
        rows.append({'Id': row.Id, 'Target': int(preds[int(row.patch_index)])})
submission = pd.DataFrame(rows, columns=['Id', 'Target'])
submission = target_df[['Id']].merge(submission, on='Id', how='left')
if len(submission) != len(target_df):
    raise AssertionError('row count mismatch')
if submission['Target'].isna().any():
    raise AssertionError('missing predictions')
submission['Target'] = submission['Target'].astype(int)
if not submission['Target'].between(0, 1023).all():
    raise AssertionError('Target outside 0..1023')

sub_root = WORKING / 'submission_v5.csv'
sub_out = OUT / 'submission_v5.csv'
submission.to_csv(sub_root, index=False)
submission.to_csv(sub_out, index=False)

sample_dir = OUT / 'sample_generate'
sample_dir.mkdir(exist_ok=True)
for image_id in target_df['image_id'].drop_duplicates().head(CFG.sample_images):
    src = next((p for p in test_images if p.stem == image_id or p.name == image_id), None)
    if src is not None:
        shutil.copy2(src, sample_dir / src.name)

progress.update({
    'stage':'submission_ready',
    'submission_rows':len(submission),
    'submission_source':progress['predictor_source'],
    'sample_generate_images':len(list(sample_dir.glob('*'))),
})
progress['artifacts'] += [str(sub_root), str(sub_out), str(sample_dir)]
save_progress()
print({'submission': str(sub_root), 'rows': len(submission), 'source': progress['submission_source'], 'sample_generate_images': progress['sample_generate_images']})



## Bundle compact outputs



In [ ]:
# Keep downloads small; code CSVs are reproducible and can be huge with all train.
for p in [train_codes_csv, test_codes_csv, train_target_csv, test_target_csv]:
    try:
        Path(p).unlink(missing_ok=True)
    except Exception as exc:
        progress['warnings'].append(f'could not delete {p}: {exc!r}')

zip_path = WORKING / 'aquatic_prior_v5_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in OUT.rglob('*'):
        if p.is_file() and p.name not in {'train_codes_v5.csv', 'test_codes_v5.csv'}:
            zf.write(p, p.relative_to(OUT.parent))
    zf.write(WORKING / 'submission_v5.csv', 'submission_v5.csv')
progress['status'] = 'complete'
progress['stage'] = 'done'
progress['artifacts'].append(str(zip_path))
save_progress()
print(json.dumps({
    'status': progress['status'],
    'submission_rows': progress['submission_rows'],
    'submission_source': progress['submission_source'],
    'mean_weighted_visible_score': progress.get('mean_weighted_visible_score'),
    'sample_generate_images': progress['sample_generate_images'],
    'train_images_used': progress['train_images_used'],
    'zip': str(zip_path),
}, indent=2))

